In [ ]:
import librosa
import numpy as np
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

print("All libraries imported successfully!")

In [ ]:
def extract_features(filepath):
    y, sr = librosa.load(filepath, duration=3, offset=0.5)
    mfcc   = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40).T, axis=0)
    chroma = np.mean(librosa.feature.chroma_stft(y=y, sr=sr).T, axis=0)
    mel    = np.mean(librosa.feature.melspectrogram(y=y, sr=sr).T, axis=0)
    return np.hstack([mfcc, chroma, mel])

def parse_and_extract(filepath):
    filename = filepath.split("\\")[-1].replace(".wav", "")
    parts    = filename.split("-")
    features = extract_features(filepath)
    return {
        "features": features,
        "emotion":  int(parts[2]),
        "actor":    int(parts[6])
    }

print("Functions defined!")

In [ ]:
all_files = glob.glob(r"F:\Study\IAT360\Week3-ClassicML\data\Actor_*\*.wav")
print(f"Found {len(all_files)} files, extracting features (this may take a few minutes)...")

data    = [parse_and_extract(f) for f in all_files]
X       = np.array([d["features"] for d in data])
y       = np.array([d["emotion"]  for d in data])
actors  = np.array([d["actor"]    for d in data])

print(f"Done! Total samples: {len(X)}")
print(f"Feature shape per sample: {X.shape}")

In [ ]:
ravdess_mask = actors <= 24
your_mask    = actors == 25

X_ravdess = X[ravdess_mask]
y_ravdess = y[ravdess_mask]
X_your    = X[your_mask]
y_your    = y[your_mask]

X_train, X_test_rav, y_train, y_test_rav = train_test_split(
    X_ravdess, y_ravdess, test_size=0.2, random_state=42
)

print(f"Training samples:         {len(X_train)}")
print(f"RAVDESS test samples:     {len(X_test_rav)}")
print(f"Your voice test samples:  {len(X_your)}")

In [ ]:
scaler = StandardScaler()

# Scaled versions
X_train_scaled     = scaler.fit_transform(X_train)
X_test_rav_scaled  = scaler.transform(X_test_rav)
X_your_scaled      = scaler.transform(X_your)

# Unscaled versions (same data, no scaling)
X_train_unscaled    = X_train
X_test_rav_unscaled = X_test_rav
X_your_unscaled     = X_your

print("Scaling done!")

In [ ]:
models = {
    "SVM":           SVC(kernel='rbf', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN":           KNeighborsClassifier(n_neighbors=5)
}

results = []

for model_name, model in models.items():
    for scale_name, X_tr, X_te_rav, X_te_your in [
        ("Scaled",   X_train_scaled,   X_test_rav_scaled,   X_your_scaled),
        ("Unscaled", X_train_unscaled, X_test_rav_unscaled, X_your_unscaled)
    ]:
        model.fit(X_tr, y_train)

        acc_rav  = accuracy_score(y_test_rav, model.predict(X_te_rav))
        acc_your = accuracy_score(y_your,     model.predict(X_te_your))

        print(f"\n=== {model_name} ({scale_name}) ===")
        print(f"RAVDESS test accuracy:    {acc_rav:.4f}")
        print(f"Your voice test accuracy: {acc_your:.4f}")

        results.append({
            "Model":                model_name,
            "Scaling":              scale_name,
            "RAVDESS Accuracy":     round(acc_rav,  4),
            "Your Voice Accuracy":  round(acc_your, 4)
        })

results_df = pd.DataFrame(results)
print("\n=== Full Summary Table ===")
print(results_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.barplot(data=results_df, x="Model", y="RAVDESS Accuracy",
            hue="Scaling", ax=axes[0])
axes[0].set_title("RAVDESS Test Accuracy")
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Accuracy")

sns.barplot(data=results_df, x="Model", y="Your Voice Accuracy",
            hue="Scaling", ax=axes[1])
axes[1].set_title("Your Voice Test Accuracy")
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("Accuracy")

plt.suptitle("Model Comparison: Scaled vs Unscaled", fontsize=14)
plt.tight_layout()
plt.savefig("results_comparison.png", dpi=150)
plt.show()
print("Plot saved as results_comparison.png")

In [ ]:
emotion_names = ["neutral","calm","happy","sad",
                 "angry","fearful","disgust","surprised"]

# Train best model (SVM Scaled)
best_model = SVC(kernel='rbf', random_state=42)
best_model.fit(X_train_scaled, y_train)

print("=== RAVDESS Test Set — Detailed Report ===")
print(classification_report(
    y_test_rav,
    best_model.predict(X_test_rav_scaled),
    target_names=emotion_names
))

print("=== Your Voice Test Set — Detailed Report ===")
print(classification_report(
    y_your,
    best_model.predict(X_your_scaled),
    target_names=emotion_names
))